In [1]:
import asyncio
import pandas as pd
import random
import os
from twikit import Client

INPUT_CSV = 'softcone_streamer_dataset1.csv'
OUTPUT_CSV = 'twitter_followers_result.csv'

USERNAME = 'bibibig1021'
EMAIL = 'bibibig040@gmail.com'
PASSWORD = 'whdlro2Qma!'

async def main():
    print("📁 CSV 데이터 로드 중...")
    df = pd.read_csv(INPUT_CSV)
    streamer_list = df['streamer_name'].dropna().unique().tolist()
    
    # ==========================================
    # 🎯 테스트 범위 설정 (여기서 수정하세요!)
    # ==========================================
    streamer_list = streamer_list[:60] 
    print(f"✅ 파일럿 테스트 모드: 총 {len(streamer_list)}명 타겟 확인!\n")

    client = Client('ko-KR')

    # ---------------------------------------------------------
    # 💡 [임시 수정] 쿠키 수동 주입 단계
    # 브라우저에서 가져온 값을 아래 따옴표 안에 넣어주세요!
    # ---------------------------------------------------------
    my_auth_token = 'aa1032655897524d4793f3af439a7816ff4b8e30'
    my_ct0 = '41bf122b6131f8b49b4f1c26ca00ef1718a069bbfd84c544de92051320c1a7f9130f3ed95b66b8c600f2044c608bcebb3acc5638113581061273d9b869abee27367e5c0d359dde34fa9f07a66a0e9129'

    # 쿠키를 딕셔너리 형태로 만듭니다.
    manual_cookies = {
        'auth_token': my_auth_token,
        'ct0': my_ct0
    }

    # 라이브러리에 쿠키를 강제로 먹입니다(?)
    client.set_cookies(manual_cookies)

    # ❗중요: 주입한 쿠키를 'cookies.json' 파일로 저장합니다.
    # 이렇게 한 번 저장해두면, 나중에 코드를 원상복구해도 이 파일을 읽어서 로그인합니다!
    client.save_cookies('cookies.json')
    print("✅ 쿠키 수동 주입 및 'cookies.json' 저장 완료! 이제 수집을 시작합니다.")
    # ---------------------------------------------------------

    results = []
    # (이후 크롤링 로직은 그대로 두시면 됩니다)
    
    
    # ---------------- 여기서부터 ----------------
    print("\n🕵️ 라이브러리 정상 작동 여부 테스트 시작...\n")
    
    for i, name in enumerate(streamer_list):
        try:
            # 1. 특정 아이디 직접 조회 (검색 기능이 고장 났는지 확인용)
            print(f"🔍 [테스트] 'Twitter' 공식 계정 정보 조회를 시도합니다...")
            user = await client.get_user_by_screen_name('Shirayukihina_')
            
            # 2. 성공 시 출력
            print(f"🟢 [SUCCESS] 라이브러리가 정상적으로 데이터를 가져왔습니다!")
            print(f"📊 Twitter 계정 팔로워 수: {user.followers_count:,}명")
            
            # 테스트 성공 시 하나만 확인하고 루프 종료
            break

        except Exception as e:
            # 3. 실패 시 에러 내용 상세 출력
            print(f"🔴 [FAILURE] 여전히 에러가 발생합니다: {e}")
            
            if "KEY_BYTE" in str(e):
                print("\n⚠️ [진단 결과] 현재 twikit 라이브러리가 트위터의 최신 보안 패치를 뚫지 못하고 있습니다.")
                print("선샌니의 잘못이 아니라 라이브러리 도구가 고장 난 상태예요!")
                print("터미널에서 'uv add --upgrade twikit' 명령어로 업데이트를 시도해보시고, 안 되면 반나절 정도 기다려야 합니다.")
            break
    # ---------------- 여기까지만 붙여넣으세요! ----------------

    # (이 아래에 있는 if results: 로직은 그대로 두시면 됩니다!)
    if results:
        temp_df = pd.DataFrame(results)
        # ... 후략
    

    # 💾 마지막 남은 데이터 최종 저장
    if results:
        temp_df = pd.DataFrame(results)
        file_exists = os.path.exists(OUTPUT_CSV)
        temp_df.to_csv(OUTPUT_CSV, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')
        
    print("\n🏁 파일럿 테스트 완료! 'twitter_followers_result.csv'를 확인해보세요!")

# 실행 (Jupyter Notebook에서는 await main() 으로 실행)
await main()

📁 CSV 데이터 로드 중...
✅ 파일럿 테스트 모드: 총 60명 타겟 확인!

✅ 쿠키 수동 주입 및 'cookies.json' 저장 완료! 이제 수집을 시작합니다.

🕵️ 라이브러리 정상 작동 여부 테스트 시작...

🔍 [테스트] 'Twitter' 공식 계정 정보 조회를 시도합니다...
🔴 [FAILURE] 여전히 에러가 발생합니다: Couldn't get KEY_BYTE indices

⚠️ [진단 결과] 현재 twikit 라이브러리가 트위터의 최신 보안 패치를 뚫지 못하고 있습니다.
선샌니의 잘못이 아니라 라이브러리 도구가 고장 난 상태예요!
터미널에서 'uv add --upgrade twikit' 명령어로 업데이트를 시도해보시고, 안 되면 반나절 정도 기다려야 합니다.

🏁 파일럿 테스트 완료! 'twitter_followers_result.csv'를 확인해보세요!


In [ ]:
for i, name in enumerate(streamer_list):
        try:
            users = await client.search_user(name)
            
            if users:
                target_user = None

                # [1순위] 이름 완벽 일치
                for u in users:
                    if u.name == name:
                        target_user = u
                        break
                
                # [2순위] 소개글에 버튜버/스트리머 키워드가 있는 사람
                if not target_user:
                    keywords = ['스트리머', '버튜버', 'vtuber', '방송', 'twitch', 'chzzk', 'youtube', '치지직']
                    for u in users:
                        bio = u.description.lower() if u.description else ""
                        if name in u.name and any(kw in bio for kw in keywords):
                            target_user = u
                            break

                # [3순위] 팔로워 100명 이상인 첫 번째 사람
                if not target_user:
                    for u in users:
                        if u.followers_count > 100:
                            target_user = u
                            break
                
                # 🎯 최종 타겟 매칭 및 [200 OK] 출력
                if target_user:
                    data = {
                        'streamer_name': name,
                        'twitter_id': target_user.screen_name,
                        'followers': target_user.followers_count,
                        'collected_at': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
                    }
                    results.append(data)
                    print(f"🟢 [200 OK] [{i+1}/{len(streamer_list)}] {name} 매칭! (@{target_user.screen_name}): {target_user.followers_count:,}명")
                else:
                    print(f"⚪ [SKIP] [{i+1}/{len(streamer_list)}] '{name}': 조건에 맞는 계정 없음")

            else:
                print(f"⚪ [404 Not Found] [{i+1}/{len(streamer_list)}] '{name}': 검색 결과 없음")

        except Exception as e:
            error_msg = str(e).lower()
            
            # 🚨 찐 에러와 단순 검색 에러 구분해서 출력
            if 'not found' in error_msg or '404' in error_msg:
                 print(f"⚪ [404 Not Found] [{i+1}/{len(streamer_list)}] '{name}': 존재하지 않는 유저")
            else:
                 # 차단(403) 또는 요청 초과(429) 등 찐 에러 발생 시
                 if 'forbidden' in error_msg or '403' in error_msg:
                     print(f"🔴 [403 Forbidden] [{i+1}/{len(streamer_list)}] 트위터 차단 접근 거부! ({name})")
                 elif 'too many' in error_msg or '429' in error_msg:
                     print(f"🔴 [429 Too Many Requests] [{i+1}/{len(streamer_list)}] 속도 제한 걸림! ({name})")
                 else:
                     print(f"🔴 [ERROR] [{i+1}/{len(streamer_list)}] 알 수 없는 오류 ({name}): {e}")
                     
                 print("🚨 15분간 비상 대기 모드 돌입!")
                 
                 # 비상 저장
                 if results:
                     temp_df = pd.DataFrame(results)
                     file_exists = os.path.exists(OUTPUT_CSV)
                     temp_df.to_csv(OUTPUT_CSV, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')
                     results = []
                     
                 await asyncio.sleep(900)
                 continue 

        # ==========================================
        # 🛡️ 공통 스텔스 휴식 로직 
        # ==========================================
        await asyncio.sleep(random.uniform(2, 5))

        # 50명마다 소규모 휴식
        if (i + 1) % 50 == 0:
            print(f"\n⏳ [소규모 휴식] {i+1}명 처리 완료. 1~2분 숨 고르기 및 중간 저장...")
            await asyncio.sleep(random.uniform(60, 120))
            if results: 
                temp_df = pd.DataFrame(results)
                file_exists = os.path.exists(OUTPUT_CSV)
                temp_df.to_csv(OUTPUT_CSV, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')
                results = [] 

        # 500명마다 대규모 휴식 (테스트 모드에서는 도달 안 하겠지만 놔둡니다!)
        if (i + 1) % 500 == 0:
            print(f"\n🛑 [대규모 휴식] {i+1}명 달성! 15~30분 딥슬립 💤...")
            await asyncio.sleep(random.uniform(900, 1800))